In [3]:
from pandas import DataFrame, read_sql_query
from datetime import datetime
from pytz import timezone
from sqlalchemy.dialects.postgresql.base import PGDialect; PGDialect._get_server_version_info = lambda * args: (9, 2)
from dataHub import dataHub

dh = dataHub()
db_connection = dh.db_connect('postgre')
# db_connection = dh.db_connect('cockroach')
fty_con = dh.fty_api_con()

In [4]:
def custom_prelog(eval_string, table_name):
    try:
        eval(eval_string)
        success = True  
        error_message = None
    except Exception as e:
        success = False
        error_message = str(e)
        print('--------------------- NOT UPDATED:', table_name)
    finally:
        DataFrame(
            data = {
                'table_name': table_name, 
                'process_date': datetime.now(timezone('NZ')), 
                'successful_run': success, 
                'error_message': error_message
            }, 
            index=[0]
        ).to_sql('update_log', db_connection, schema='util', index=False, if_exists='append')

In [5]:
# Obtain update_schedule filtering on US Eastern Time
us_eastern_time = datetime.now(timezone('US/Eastern')).strftime('%Y-%m-%d')
us_eastern_time = '2024-03-05'
update_schedule = read_sql_query("SELECT * FROM util.update_schedule WHERE run_period_start <= '{}' AND run_period_end >= '{}'".format(us_eastern_time, us_eastern_time), db_connection)

In [9]:
update_schedule

,table_name,update_schedule,associated_function,function_arguments,run_period_start,run_period_end
0,nba.transaction_log,daily,get_transactions,db_connection,2023-10-05,2024-06-06
1,nba.player_season_stats,weekly,get_player_season_stats,db_connection,2023-10-05,2024-06-06
2,nba.player_game_log,daily,get_player_game_log,db_connection,2023-10-05,2024-06-06
3,fty.free_agents,daily,fty_get_free_agents,"fty_con.free_agents(size=1000), db_connection",2023-10-05,2024-06-06


In [7]:
# Loop through update schedule and update objects accordingly
for _, row in update_schedule.iterrows():

    # Create evaluation string & handle functions that only need to be run weekly
    eval_string = ''.join(['dh.', row['associated_function'], '(', row['function_arguments'], ')'])
    if ((row[['update_schedule']].str.contains('weekly')[0]) and (datetime.now(timezone('NZ')).weekday() != 0)):
        eval_string = None

    if (eval_string is not None):
        custom_prelog(eval_string, row['table_name'])


--------------------- player_game_log
player: 0 / 535
player: 50 / 535
player: 100 / 535
player: 150 / 535
player: 200 / 535
player: 250 / 535
player: 300 / 535
player: 350 / 535
player: 400 / 535
player: 450 / 535
player: 500 / 535
player: 534 / 535
player_game_log has been updated to: 2023-09-10
free_agents has been updated


In [8]:
# Print failed objects
current_date = datetime.now(timezone('NZ')).strftime('%Y-%m-%d')
failed_objects = read_sql_query("SELECT table_name FROM util.update_log WHERE successful_run = False AND process_date::DATE = '{}'".format(current_date), db_connection)

if len(failed_objects) > 0:
    raise Exception(print(current_date, '\nFailed objects:', failed_objects['table_name'].to_list()))
else:
    print(current_date, '\nAll tables successfully updated.')

2023-09-12 
Failed objects: ['nba.transaction_log']


Exception: None